# RobustRAG Table I reproduction

This notebook reproduces the Mistral-7B keyword-aggregation results from Table I of *Certifiably Robust RAG against Retrieval Corruption*. It keeps the two paper versions separate:

- **Legacy/code profile:** `alpha=0.3`, `beta=3`
- **Current attached-paper profile:** `alpha=0.2`, `beta=3`

Use an **A100 or L4** Colab runtime and run cells in order. The paper used an A100; L4 results are recorded as a hardware-deviated reproduction. The notebook downloads the exact upstream commit into ephemeral `/content`, verifies dataset checksums, reuses cached LLM responses, and saves every experiment in a new directory under the shared Drive.

The upstream repository has no license file, so its source is not copied into this project. It is fetched directly from the authors and pinned to commit `9bc35b2fa5fa7d1088383b2789ec19a512d316b1`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, importlib
sys.dont_write_bytecode = True
PROJ = "/content/drive/MyDrive/colab/adversarial_rag"
SHARED = "/content/drive/MyDrive/colab/adversarial_rag-shared"
os.chdir(PROJ)
sys.path.insert(0, PROJ)

# LiteLLM 1.35.1 is no longer available from PyPI. The Mistral path only
# needs LiteLLM to satisfy the upstream module-level import; it does not call it.
from pathlib import Path
requirements = Path("requirements.txt").read_text().splitlines()
requirements = [line for line in requirements if not line.startswith("litellm==")]
Path("/tmp/requirements-colab.txt").write_text("\n".join(requirements) + "\n")

!pip install -q -r /tmp/requirements-colab.txt
# Install the available compatibility release with its declared dependencies.
!pip install -q "litellm==1.14.4"

import litellm
from litellm import batch_completion
print("LiteLLM compatibility import succeeded.")

: 

: 

## Personal output folder and hardware check

Enter your name when prompted. The notebook is shared, but experiment outputs remain separated as required by `ONBOARDING.md`.

In [ ]:
from pathlib import Path
import subprocess

OWNER = input("Output folder name (for example, manya or connor): ").strip()
if not OWNER or OWNER.lower() in {"replace-me", "your-name"}:
    raise ValueError("Enter a non-empty personal output folder name.")

DATA_DIR = Path(SHARED) / "data" / "robustrag"
RUNS_DIR = Path(SHARED) / "outputs" / OWNER / "robustrag-reproduction"
UPSTREAM_DIR = Path("/content/robustrag-upstream")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    text=True,
).strip()
print("GPU:", gpu)
supported_gpus = ("A100", "L4")
if not any(gpu_name in gpu for gpu_name in supported_gpus):
    raise RuntimeError("Select an A100 or L4 GPU runtime.")
if "L4" in gpu:
    print("Note: the paper used an A100; this run will be labeled as an L4 reproduction.")
print("Data:", DATA_DIR)
print("Runs:", RUNS_DIR)

: 

## Fetch and verify the authors' code and data

This cell is safe to rerun. It reuses exact matches and refuses to overwrite anything different. The JSON datasets stay in the shared data directory, outside git.

In [ ]:
import hashlib
import shutil
import tempfile
import urllib.request

UPSTREAM_URL = "https://github.com/inspire-group/RobustRAG.git"
UPSTREAM_COMMIT = "9bc35b2fa5fa7d1088383b2789ec19a512d316b1"
DATA_BASE_URL = f"https://raw.githubusercontent.com/inspire-group/RobustRAG/{UPSTREAM_COMMIT}/data"
DATASETS = {
    "realtimeqa": ("realtimeqa.json", "0924fd01734f90e7910d6cb5802d387aaf5811eeba650667a8e72d6b94d7941e"),
    "open_nq": ("open_nq.json", "32a71eab0a71f67e8f07ca8fddf8d9ac4d18c841d92df3f77a0e0cfa11ad8f11"),
}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def git_head(path):
    return subprocess.check_output(
        ["git", "-C", str(path), "rev-parse", "HEAD"], text=True
    ).strip()

def fetch_upstream_code():
    if UPSTREAM_DIR.exists():
        if git_head(UPSTREAM_DIR) == UPSTREAM_COMMIT:
            print("Reusing verified upstream code:", UPSTREAM_DIR)
            return
        raise RuntimeError(f"Refusing to overwrite unexpected checkout: {UPSTREAM_DIR}")
    temporary_parent = Path(tempfile.mkdtemp(prefix="robustrag-code-", dir="/content"))
    checkout = temporary_parent / "checkout"
    try:
        subprocess.run(["git", "init", str(checkout)], check=True)
        subprocess.run(["git", "-C", str(checkout), "remote", "add", "origin", UPSTREAM_URL], check=True)
        subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", UPSTREAM_COMMIT], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"], check=True)
        if git_head(checkout) != UPSTREAM_COMMIT:
            raise RuntimeError("Downloaded code does not match the pinned commit.")
        checkout.replace(UPSTREAM_DIR)
        print("Created verified upstream checkout:", UPSTREAM_DIR)
    finally:
        if temporary_parent.exists():
            shutil.rmtree(temporary_parent)

def fetch_dataset(filename, expected_hash):
    destination = DATA_DIR / filename
    if destination.exists():
        actual_hash = file_sha256(destination)
        if actual_hash == expected_hash:
            print("Reusing verified dataset:", destination)
            return
        raise RuntimeError(f"Refusing to overwrite {destination}; checksum is {actual_hash}")
    descriptor, temporary_name = tempfile.mkstemp(prefix=f".{filename}.", dir=DATA_DIR)
    os.close(descriptor)
    temporary = Path(temporary_name)
    try:
        urllib.request.urlretrieve(f"{DATA_BASE_URL}/{filename}", temporary)
        actual_hash = file_sha256(temporary)
        if actual_hash != expected_hash:
            raise RuntimeError(f"Checksum mismatch for {filename}: {actual_hash}")
        temporary.replace(destination)
        print("Saved verified dataset:", destination)
    finally:
        if temporary.exists():
            temporary.unlink()

subprocess.run([sys.executable, "-m", "nltk.downloader", "stopwords"], check=True)
fetch_upstream_code()
for filename, expected_hash in DATASETS.values():
    fetch_dataset(filename, expected_hash)

## Reproduction runner

The functions below run the authors' unmodified `main.py` in temporary local storage, then copy logs, responses, and caches into a new timestamped shared-output directory. Earlier runs and caches are never modified.

In [ ]:
import importlib.metadata
import json
import platform
import re
from datetime import datetime, timezone

PROFILES = {
    "legacy_v1": {
        "alpha": 0.3, "beta": 3.0,
        "expected": {
            "realtimeqa": {"no_rag": 8, "vanilla": 69, "keyword": 59, "certified": 45},
            "open_nq": {"no_rag": 30, "vanilla": 61, "keyword": 54, "certified": 47},
        },
    },
    "current_v2": {
        "alpha": 0.2, "beta": 3.0,
        "expected": {
            "realtimeqa": {"no_rag": 8, "vanilla": 69, "keyword": 71, "certified": 38},
            "open_nq": {"no_rag": 30, "vanilla": 61, "keyword": 61, "certified": 26},
        },
    },
}

def package_versions():
    names = ["accelerate", "joblib", "litellm", "nltk", "numpy", "openai", "scikit-learn", "spacy", "torch", "transformers"]
    versions = {}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

def upstream_command(condition, dataset, profile):
    common = [sys.executable, str(UPSTREAM_DIR / "main.py"), "--model_name", "mistral7b", "--dataset_name", dataset, "--save_response", "--use_cache"]
    if condition == "no_rag":
        return common + ["--top_k", "0", "--defense_method", "none", "--corruption_size", "0"]
    if condition == "vanilla":
        return common + ["--top_k", "10", "--defense_method", "none", "--corruption_size", "0"]
    if condition == "keyword":
        return common + ["--top_k", "10", "--defense_method", "keyword", "--corruption_size", "1", "--alpha", str(profile["alpha"]), "--beta", str(profile["beta"]), "--no_vanilla"]
    raise ValueError(f"Unknown condition: {condition}")

def stream_command(command, cwd, transcript):
    print("Running:", " ".join(command), flush=True)
    with Path(transcript).open("w") as output:
        process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            output.write(line)
        return_code = process.wait()
    if return_code:
        raise RuntimeError(f"Upstream command failed with exit code {return_code}")

def parse_log(log_path, condition, sample_size):
    text = Path(log_path).read_text()
    def count(label):
        matches = re.findall(rf"{re.escape(label)}:\s*(\d+)", text)
        return int(matches[-1]) if matches else None
    counts = {
        "undefended_correct": count("undefended_corr_cnt"),
        "defended_correct": count("defended_corr_cnt"),
        "certified_correct": count("certify_cnt"),
    }
    primary = counts["undefended_correct"] if condition in {"no_rag", "vanilla"} else counts["defended_correct"]
    if primary is None:
        raise RuntimeError(f"Could not parse metrics from {log_path}")
    metrics = {"sample_size": sample_size, "correct_count": primary, "accuracy": primary / sample_size, "counts": counts}
    if condition == "keyword":
        metrics["certified_accuracy"] = counts["certified_correct"] / sample_size
    return metrics

def copy_runtime_artifacts(runtime, run_dir):
    for name in ["cache", "log", "result", "result_certify"]:
        source = runtime / name
        if source.exists():
            shutil.copytree(source, run_dir / name, dirs_exist_ok=False)

def run_table1(profile_name, dataset, conditions=("keyword",), limit=100, cache_from=None):
    if profile_name not in PROFILES or dataset not in DATASETS:
        raise ValueError("Unknown profile or dataset.")
    if not 1 <= limit <= 100:
        raise ValueError("limit must be between 1 and 100.")
    if len(set(conditions)) != len(conditions):
        raise ValueError("Conditions must be unique.")
    if git_head(UPSTREAM_DIR) != UPSTREAM_COMMIT:
        raise RuntimeError("The upstream checkout changed.")

    profile = PROFILES[profile_name]
    filename, expected_hash = DATASETS[dataset]
    source_dataset = DATA_DIR / filename
    if file_sha256(source_dataset) != expected_hash:
        raise RuntimeError(f"Dataset checksum mismatch: {source_dataset}")

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = RUNS_DIR / f"table1-{profile_name}-{dataset}-{timestamp}"
    run_dir.mkdir(parents=True, exist_ok=False)
    manifest = {
        "status": "running", "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "profile": profile_name, "dataset": dataset, "dataset_sha256": expected_hash,
        "upstream_commit": UPSTREAM_COMMIT, "sample_limit": limit, "conditions": list(conditions),
        "parameters": {"model": "mistralai/Mistral-7B-Instruct-v0.2", "top_k": 10, "group_size": 1, "corruption_size": 1, "alpha": profile["alpha"], "beta": profile["beta"], "greedy_decoding": True},
        "expected_full_sample_counts": profile["expected"][dataset],
        "environment": {"python": sys.version, "platform": platform.platform(), "gpu": gpu, "hardware_matches_paper": "A100" in gpu, "packages": package_versions()},
        "commands": [],
    }
    manifest_path = run_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2))

    try:
        with tempfile.TemporaryDirectory(prefix="robustrag-run-", dir="/content") as temporary:
            runtime = Path(temporary)
            records = json.loads(source_dataset.read_text())[:limit]
            (runtime / "data").mkdir()
            (runtime / "data" / filename).write_text(json.dumps(records))
            if cache_from is not None:
                cache_source = Path(cache_from)
                if (cache_source / "cache").is_dir():
                    cache_source = cache_source / "cache"
                shutil.copytree(cache_source, runtime / "cache")
            metrics = {}
            try:
                for condition in conditions:
                    old_logs = set((runtime / "log").glob("*.log"))
                    command = upstream_command(condition, dataset, profile)
                    manifest["commands"].append(command)
                    manifest_path.write_text(json.dumps(manifest, indent=2))
                    stream_command(command, runtime, run_dir / f"{condition}.txt")
                    new_logs = set((runtime / "log").glob("*.log")) - old_logs
                    if not new_logs:
                        raise RuntimeError("The upstream run did not create a new log.")
                    log_path = max(new_logs, key=lambda path: path.stat().st_mtime_ns)
                    condition_metrics = parse_log(log_path, condition, len(records))
                    if len(records) == 100:
                        expected = profile["expected"][dataset]
                        condition_metrics["expected_correct_count"] = expected[condition]
                        condition_metrics["matches_expected_correct_count"] = condition_metrics["correct_count"] == expected[condition]
                        if condition == "keyword":
                            condition_metrics["expected_certified_count"] = expected["certified"]
                            condition_metrics["matches_expected_certified_count"] = condition_metrics["counts"]["certified_correct"] == expected["certified"]
                    metrics[condition] = condition_metrics
                    (run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
            finally:
                copy_runtime_artifacts(runtime, run_dir)
        manifest["status"] = "complete"
        manifest["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
        manifest_path.write_text(json.dumps(manifest, indent=2))
        print("Completed:", run_dir)
        print(json.dumps(metrics, indent=2))
        return run_dir
    except Exception:
        manifest["status"] = "failed"
        manifest["failed_at_utc"] = datetime.now(timezone.utc).isoformat()
        manifest_path.write_text(json.dumps(manifest, indent=2))
        raise

## 1. Two-question smoke test

This downloads Mistral-7B and verifies that the legacy pipeline completes before spending time on the full sample.

In [ ]:
smoke_run = run_table1(
    profile_name="legacy_v1",
    dataset="realtimeqa",
    conditions=("keyword",),
    limit=2,
)

## 2. Full RealtimeQA reproduction

Run the legacy/code profile first. The current-paper run then copies its cache, so isolated responses are not regenerated.

In [ ]:
legacy_rqa_run = run_table1(
    profile_name="legacy_v1",
    dataset="realtimeqa",
    conditions=("no_rag", "vanilla", "keyword"),
)

In [ ]:
current_rqa_run = run_table1(
    profile_name="current_v2",
    dataset="realtimeqa",
    conditions=("no_rag", "vanilla", "keyword"),
    cache_from=legacy_rqa_run,
)

## 3. Full Natural Questions reproduction

Again, run the legacy profile first and reuse its cache for the current-paper profile. The upstream file contains 500 questions, but Table I and this notebook use its first 100.

In [ ]:
legacy_nq_run = run_table1(
    profile_name="legacy_v1",
    dataset="open_nq",
    conditions=("no_rag", "vanilla", "keyword"),
)

In [ ]:
current_nq_run = run_table1(
    profile_name="current_v2",
    dataset="open_nq",
    conditions=("no_rag", "vanilla", "keyword"),
    cache_from=legacy_nq_run,
)

## 4. Compare observed and expected counts

An exact match is ideal because these are the same fixed 100 questions. If only the current profile differs, report the repository/paper revision gap rather than tuning parameters until it matches.

In [ ]:
completed_runs = {
    "legacy_rqa": legacy_rqa_run,
    "current_rqa": current_rqa_run,
    "legacy_nq": legacy_nq_run,
    "current_nq": current_nq_run,
}
for label, run_path in completed_runs.items():
    print(f"\n{label}: {run_path}")
    print((Path(run_path) / "metrics.json").read_text())

## Finish

After confirming that all files have finished writing, unmount Drive. Then run `./pull.sh` locally to download the shared outputs. Before committing this notebook, use **Edit → Clear all outputs** so model logs and environment details are not stored in git.

In [ ]:
drive.flush_and_unmount()